# Local conformational variation with RFdiffusion3

This notebook adapts the Binder/Refold AGP experiment into a reusable Colab workflow. It generates local conformational alternatives while keeping the selected protein sequence and the surrounding scaffold fixed.

## Quick start

1. Run the notebook from top to bottom. The setup step installs the required packages and RFD3 checkpoint.
2. In **Choose input and settings**, select **Sequence** or **PDB/mmCIF structure** and set the target residue and region to vary.
3. For a structure, leave the optional path blank and run the clearly labeled **Upload PDB or mmCIF** step; choose one `.pdb`, `.cif`, or `.mmcif` file. Sequence mode skips this step.
4. Continue through the sampling and analysis steps. Inspect the aligned ensemble in Mol*, then use the buttons at the end to download individual files or a ZIP bundle.

Implementation code is collapsed by default for a cleaner Colab interface; use Colab's **Show code** control whenever you want to inspect it. This is a visual convenience, not code protection.

The default example is the 181-residue AGP sequence used for the Asn36 experiment (`N36`, window `A30–A42`). Replace it with your sequence or a structure. For sequence-mode mutations, use one-letter notation such as `N36A` or `N36A,S38T`; the reference residue is checked before RF3 folding. For a structure-mode mutation, upload a chemically modeled/relaxed mutant structure—renaming a residue in a coordinate file is not a valid side-chain model. Include glycans or other components in the uploaded structure if they should be retained as context.

## Methodology

1. **Prepare a starting structure.** A sequence (with any requested substitutions) is folded once with RF3. A PDB/mmCIF input is used with its chain and residue numbering.
2. **Define the local region.** Choose a target residue and a residue window. Atoms outside that window are selected as fixed scaffold anchors.
3. **Generate local alternatives with partial diffusion.** RFD3 perturbs the selected region around the input structure while `select_fixed_atoms` holds the outside scaffold in place and `select_unfixed_sequence=False` keeps sequence identity fixed. `PARTIAL_T` is the noise scale in Å—not elapsed time or a count of steps. `EXPOSURE_CONDITION` maps to exposed-surface conditioning; `LOOP_BIAS` maps to `is_non_loopy` (`False` requests more loops, `True` fewer). These are model guidance, not guarantees. The RFD3 input guide recommends starting conservatively; this notebook defaults to 12 Å.
4. **Assess candidates.** The notebook samples independent structures, annotates secondary structure with P-SEA, and checks for large adjacent Cα gaps and any clash count reported in RFD3 metadata. `REQUIRE_TARGET_NONHELICAL` is a post-generation acceptance filter requiring a coil call at the target; it does not directly force the model to produce one. This is a filter, not a physics-based energy ranking; all generated candidates remain available for inspection.
5. **Align for comparison.** By default, each model is rigidly Kabsch-aligned to the original using common protein Cα atoms outside the selected window. This removes overall pose differences without deforming the structures, making the local change easier to see.

> These are model-generated structural hypotheses, not a molecular-dynamics trajectory, a measured unfolding pathway, or equilibrium populations. Do not interpret the fraction of accepted samples as a thermodynamic probability. Inspect geometry and validate candidates independently before drawing mechanistic conclusions.

For definitions of `partial_t`, fixed-atom selections, exposure conditioning, and loop guidance, see the [official RFD3 input specification](https://github.com/RosettaCommons/foundry/blob/production/models/rfd3/docs/input.md).

In [ ]:
#@title Install dependencies and the RFD3 checkpoint { display-mode: "form" }
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ.setdefault('CCD_MIRROR_PATH', '')
os.environ.setdefault('PDB_MIRROR_PATH', '')
# Some Colab CUDA images expose a cuEquivariance binary incompatible with the selected GPU.
# RF3's vanilla PyTorch attention is slower but portable and avoids the CUDA symbol failure.
os.environ['DISABLE_CUEQUIVARIANCE'] = '1'
print('cuEquivariance disabled for Colab compatibility; RF3 will use vanilla PyTorch attention.')
WORKDIR = Path('/content/local_conformational_ensemble')
WORKDIR.mkdir(parents=True, exist_ok=True)

def run_checked(command):
    print('$', ' '.join(str(item) for item in command))
    subprocess.check_call(command)

if importlib.util.find_spec('rfd3') is None:
    print('Installing rc-foundry (AtomWorks + RFD3 runtime)...')
    run_checked([sys.executable, '-m', 'pip', 'install', '-q', 'rc-foundry[all]'])
else:
    print('rc-foundry/RFD3 Python package is already available.')

if importlib.util.find_spec('biotite') is None or importlib.util.find_spec('pandas') is None:
    run_checked([sys.executable, '-m', 'pip', 'install', '-q', 'biotite', 'pandas'])

foundry = shutil.which('foundry')
if foundry is None:
    raise RuntimeError('The foundry command was not found after installation. Restart the Colab runtime and run this cell again.')

rfd3_marker = Path('/content/.glycoshape_rfd3_checkpoint_ready')
if not rfd3_marker.exists():
    run_checked([foundry, 'install', 'rfd3'])
    rfd3_marker.write_text('ok\n')
else:
    print('RFD3 checkpoint marker found; using the cached checkpoint.')

try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not available')
except Exception as exc:
    print('Could not query CUDA:', exc)

In [ ]:
#@title Step 1: Choose input and sampling settings { display-mode: "form" }
INPUT_MODE = 'Sequence' #@param ["Sequence", "PDB/mmCIF structure"]
UPLOADED_STRUCTURE_PATH = None  # Reset the upload whenever these settings are run.
PROTEIN_SEQUENCE = 'PLCANLVPVPITNATLDRITGKWFYIASAFRNEEYNKSVQEIQATFFYFTPNKTEDTIFLREYQTRQDQCIYNTTYLNVQRENGTISRYVGGQEHFAHLLILRDTKTYMLAFDVNDEKNWGLSVYADKPETTKEQLGEFYEALDCLRIPKSDVVYTDWKKDKCEPLEKQHEKERKQEEGES' #@param {type:"string"}
MUTATIONS = '' #@param {type:"string"}
STRUCTURE_PATH = '' #@param {type:"string"}
# Optional path to a structure already in this Colab runtime or mounted Drive; usually leave blank and use the upload step.
TARGET_CHAIN = 'A' #@param {type:"string"}
TARGET_RESIDUE = 36 #@param {type:"integer"}
WINDOW_MODE = 'Manual' #@param ["Manual", "Target helix plus padding"]
WINDOW_START = 30 #@param {type:"integer"}
WINDOW_END = 42 #@param {type:"integer"}
HELIX_PADDING = 1 #@param {type:"integer"}
FALLBACK_HALF_WINDOW = 4 #@param {type:"integer"}

N_VARIANTS = 16 #@param {type:"integer"}
DIFFUSION_BATCH_SIZE = 4 #@param {type:"integer"}
PARTIAL_T = 12.0 #@param {type:"number"}
STEP_SCALE = 0.95 #@param {type:"number"}
NOISE_SCALE = 1.015 #@param {type:"number"}
RFD3_SEED = 936 #@param {type:"integer"}
EXPOSURE_CONDITION = 'Window' #@param ["Window", "Target residue", "None"]
LOOP_BIAS = 'More loops' #@param ["More loops", "Fewer loops", "None"]
REQUIRE_TARGET_NONHELICAL = True #@param {type:"boolean"}
ALIGNMENT_MODE = 'Fixed scaffold outside window' #@param ["Fixed scaffold outside window", "All common protein C-alpha", "No alignment"]

print('Input mode:', INPUT_MODE)
print('Selected window:', f'{TARGET_CHAIN}{WINDOW_START}-{WINDOW_END}')
print('Variants requested:', N_VARIANTS)
print('Visualization alignment:', ALIGNMENT_MODE)

In [ ]:
#@title Step 2: Upload PDB or mmCIF (only for structure input)
from pathlib import Path

UPLOADED_STRUCTURE_PATH = None
if INPUT_MODE == 'Sequence':
    print('Sequence mode selected — no file upload is needed.')
elif STRUCTURE_PATH.strip():
    UPLOADED_STRUCTURE_PATH = Path(STRUCTURE_PATH).expanduser()
    if not UPLOADED_STRUCTURE_PATH.is_file():
        raise FileNotFoundError(f'Structure file not found: {UPLOADED_STRUCTURE_PATH}')
    if UPLOADED_STRUCTURE_PATH.suffix.lower() not in {'.pdb', '.ent', '.cif', '.mmcif'}:
        raise ValueError('Structure path must end in .pdb, .ent, .cif, or .mmcif.')
    print('Using structure already in the runtime:', UPLOADED_STRUCTURE_PATH)
else:
    from google.colab import files
    print('Select exactly one structure file: .pdb, .cif, or .mmcif.')
    uploaded_files = files.upload()
    if not uploaded_files:
        raise ValueError('No file selected. Rerun this upload step and choose one PDB/mmCIF file.')
    if len(uploaded_files) != 1:
        raise ValueError(f'Please upload exactly one structure file; received {len(uploaded_files)} files.')
    uploaded_name, uploaded_bytes = next(iter(uploaded_files.items()))
    uploaded_name = Path(uploaded_name).name
    if Path(uploaded_name).suffix.lower() not in {'.pdb', '.ent', '.cif', '.mmcif'}:
        raise ValueError(f'{uploaded_name!r} is not a supported structure file. Choose .pdb, .cif, or .mmcif.')
    UPLOADED_STRUCTURE_PATH = WORKDIR / uploaded_name
    UPLOADED_STRUCTURE_PATH.write_bytes(uploaded_bytes)
    print(f'Uploaded and ready: {UPLOADED_STRUCTURE_PATH.name} ({UPLOADED_STRUCTURE_PATH.stat().st_size:,} bytes)')

In [ ]:
#@title Step 3: Load analysis and visualization helpers
import base64
import json
import math
import re
import uuid
from dataclasses import dataclass

import biotite.structure as struc
import numpy as np
import pandas as pd
from biotite.structure.io.pdb import PDBFile
from biotite.structure.sse import annotate_sse
from IPython.display import HTML, display

AA1_TO_AA3 = {
    'A': 'ALA', 'R': 'ARG', 'N': 'ASN', 'D': 'ASP', 'C': 'CYS',
    'Q': 'GLN', 'E': 'GLU', 'G': 'GLY', 'H': 'HIS', 'I': 'ILE',
    'L': 'LEU', 'K': 'LYS', 'M': 'MET', 'F': 'PHE', 'P': 'PRO',
    'S': 'SER', 'T': 'THR', 'W': 'TRP', 'Y': 'TYR', 'V': 'VAL',
}

@dataclass(frozen=True)
class ResidueRecord:
    chain_id: str
    res_id: int
    res_name: str

def validate_sequence(sequence):
    lines = [line.strip() for line in str(sequence).splitlines() if not line.strip().startswith('>')]
    cleaned = ''.join(lines).replace(' ', '').replace('\t', '').upper()
    invalid = sorted(set(cleaned) - set(AA1_TO_AA3))
    if not cleaned:
        raise ValueError('The sequence is empty.')
    if invalid:
        raise ValueError(f'Invalid amino-acid characters: {invalid}')
    return cleaned

def parse_mutations(text):
    if not str(text).strip():
        return []
    mutations = []
    for token in re.split(r'[,;\s]+', str(text).strip()):
        match = re.fullmatch(r'([A-Z])(\d+)([A-Z])', token.upper())
        if match is None:
            raise ValueError(f'Could not parse mutation {token!r}; use one-letter notation such as N36A.')
        reference, position, mutant = match.groups()
        mutations.append((reference, int(position), mutant))
    return mutations

def apply_mutations(sequence, mutation_text):
    result = list(validate_sequence(sequence))
    applied = []
    for reference, position, mutant in parse_mutations(mutation_text):
        if position < 1 or position > len(result):
            raise ValueError(f'Mutation position {position} is outside the sequence (length {len(result)}).')
        observed = result[position - 1]
        if observed != reference:
            raise ValueError(f'Mutation {reference}{position}{mutant} does not match the sequence; observed {observed}{position}.')
        result[position - 1] = mutant
        applied.append(f'{reference}{position}{mutant}')
    return ''.join(result), applied

def load_structure_file(path):
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix in {'.cif', '.mmcif'}:
        try:
            from biotite.structure.io.pdbx import CIFFile, get_structure
        except ImportError:
            from biotite.structure.io.pdbx import PDBxFile as CIFFile, get_structure
        parsed = CIFFile.read(path)
        atom_array = get_structure(parsed, model=1)
    else:
        atom_array = PDBFile.read(path).get_structure(model=1)
    if isinstance(atom_array, struc.AtomArrayStack):
        atom_array = atom_array[0]
    return atom_array

def as_atom_array(value):
    if isinstance(value, struc.AtomArrayStack):
        return value[0]
    return value

def pdb_compatible_array(value):
    """Return a copy whose B-factors fit the fixed-width legacy PDB field."""
    array = as_atom_array(value).copy()
    if hasattr(array, 'b_factor'):
        b_factor = np.asarray(array.b_factor, dtype=float)
        valid = np.isfinite(b_factor) & (b_factor >= 0.0) & (b_factor <= 999.99)
        if not np.all(valid):
            safe_b_factor = b_factor.copy()
            safe_b_factor[~valid] = 0.0
            array.b_factor = safe_b_factor
    return array

def write_single_pdb(atom_array, path):
    pdb = PDBFile()
    pdb.set_structure(pdb_compatible_array(atom_array))
    pdb.write(path)

def residue_records(atom_array, protein_only=False):
    array = as_atom_array(atom_array)
    if protein_only:
        array = array[struc.filter_amino_acids(array)]
    starts = struc.get_residue_starts(array)
    records = []
    seen = set()
    for start in starts:
        key = (str(array.chain_id[start]), int(array.res_id[start]))
        if key in seen:
            continue
        seen.add(key)
        records.append(ResidueRecord(key[0], key[1], str(array.res_name[start])))
    return records

def protein_array(atom_array):
    array = as_atom_array(atom_array)
    return array[struc.filter_amino_acids(array)]

def residue_index(records, chain_id, res_id):
    for index, record in enumerate(records):
        if record.chain_id == str(chain_id) and record.res_id == int(res_id):
            return index
    raise ValueError(f'Residue {chain_id}{res_id} was not found.')

def residue_ids_to_ranges(residue_ids):
    values = sorted(set(int(value) for value in residue_ids))
    if not values:
        return []
    ranges = []
    start = end = values[0]
    for value in values[1:]:
        if value == end + 1:
            end = value
        else:
            ranges.append((start, end))
            start = end = value
    ranges.append((start, end))
    return ranges

def make_fixed_selection(records, chain_id, window_start, window_end):
    grouped = {}
    for record in records:
        inside_window = (record.chain_id == chain_id and window_start <= record.res_id <= window_end)
        if not inside_window:
            grouped.setdefault(record.chain_id, []).append(record.res_id)
    parts = []
    for chain in sorted(grouped):
        for start, end in residue_ids_to_ranges(grouped[chain]):
            parts.append(f'{chain}{start}' if start == end else f'{chain}{start}-{end}')
    if not parts:
        raise ValueError('The selected window covers the entire input; leave a fixed scaffold outside it.')
    return ','.join(parts)

def detect_window(atom_array, chain_id, target_residue, padding, fallback_half_window):
    array = protein_array(atom_array)
    records = residue_records(array)
    target = residue_index(records, chain_id, target_residue)
    sse = annotate_sse(array)
    if sse[target] == 'a':
        left = target
        while left > 0 and records[left - 1].chain_id == chain_id and sse[left - 1] == 'a':
            left -= 1
        right = target
        while right + 1 < len(records) and records[right + 1].chain_id == chain_id and sse[right + 1] == 'a':
            right += 1
        chain_positions = [i for i, record in enumerate(records) if record.chain_id == chain_id]
        left = max(left - int(padding), chain_positions[0])
        right = min(right + int(padding), chain_positions[-1])
        return records[left].res_id, records[right].res_id, 'detected helix'
    chain_positions = [i for i, record in enumerate(records) if record.chain_id == chain_id]
    chain_target = chain_positions.index(target)
    left = max(chain_target - int(fallback_half_window), 0)
    right = min(chain_target + int(fallback_half_window), len(chain_positions) - 1)
    return records[chain_positions[left]].res_id, records[chain_positions[right]].res_id, 'fallback window'

def disable_cuequivariance():
    """Force portable attention, including after a previous failed import in this runtime."""
    os.environ['DISABLE_CUEQUIVARIANCE'] = '1'
    foundry_module = sys.modules.get('foundry')
    if foundry_module is not None:
        foundry_module.SHOULD_USE_CUEQUIVARIANCE = False
    attention_module = sys.modules.get('rf3.model.layers.attention')
    if attention_module is not None:
        attention_module.SHOULD_USE_CUEQUIVARIANCE = False

def ensure_rf3_available():
    if importlib.util.find_spec('rf3') is None:
        run_checked([sys.executable, '-m', 'pip', 'install', '-q', 'rc-foundry[all]'])
    foundry = shutil.which('foundry')
    if foundry is None:
        raise RuntimeError('The foundry command is unavailable; rerun the setup cell.')
    marker = Path('/content/.glycoshape_rf3_checkpoint_ready')
    if not marker.exists():
        run_checked([foundry, 'install', 'rf3'])
        marker.write_text('ok\n')

def fold_sequence(sequence, example_id='sequence_input'):
    disable_cuequivariance()
    ensure_rf3_available()
    from atomworks.io.tools.inference import components_to_atom_array
    from rf3.inference_engines.rf3 import RF3InferenceEngine
    from rf3.utils.inference import InferenceInput
    components = [{'seq': sequence, 'chain_id': 'A'}]
    placeholder = components_to_atom_array(components)
    inference_input = InferenceInput.from_atom_array(placeholder, example_id=example_id)
    engine = RF3InferenceEngine(ckpt_path='rf3', verbose=False)
    result = engine.run(inputs=inference_input)[example_id][0]
    return as_atom_array(result.atom_array)

def ca_coordinates(atom_array):
    array = protein_array(atom_array)
    coordinates = {}
    for index in np.where(array.atom_name == 'CA')[0]:
        key = (str(array.chain_id[index]), int(array.res_id[index]))
        coordinates[key] = np.asarray(array.coord[index], dtype=float)
    return coordinates

def bend_angle(array_a, array_b, array_c):
    first = np.asarray(array_a) - np.asarray(array_b)
    second = np.asarray(array_c) - np.asarray(array_b)
    denominator = np.linalg.norm(first) * np.linalg.norm(second)
    if denominator == 0:
        return float('nan')
    cosine = np.clip(np.dot(first, second) / denominator, -1.0, 1.0)
    return float(np.degrees(np.arccos(cosine)))

def count_ca_chainbreaks(atom_array, threshold=4.5):
    records = residue_records(protein_array(atom_array))
    ca = ca_coordinates(atom_array)
    count = 0
    for previous, current in zip(records, records[1:]):
        if previous.chain_id != current.chain_id or current.res_id != previous.res_id + 1:
            continue
        first = ca.get((previous.chain_id, previous.res_id))
        second = ca.get((current.chain_id, current.res_id))
        if first is not None and second is not None and np.linalg.norm(second - first) > threshold:
            count += 1
    return count

def find_numeric_metric(value, names):
    names = {name.lower() for name in names}
    if isinstance(value, dict):
        for key, item in value.items():
            if str(key).lower() in names and isinstance(item, (int, float, np.integer, np.floating)) and not isinstance(item, bool):
                return float(item)
            found = find_numeric_metric(item, names)
            if found is not None:
                return found
    elif isinstance(value, (list, tuple)):
        for item in value:
            found = find_numeric_metric(item, names)
            if found is not None:
                return found
    return None

def analyze_structure(atom_array, original_ca, metadata, chain_id, target_residue, window_start, window_end):
    array = protein_array(atom_array)
    records = residue_records(array)
    sse = annotate_sse(array)
    code_by_residue = {(record.chain_id, record.res_id): (sse[i] or '?') for i, record in enumerate(records)}
    window_codes = [code_by_residue.get((chain_id, residue), '?') for residue in range(window_start, window_end + 1)]
    target_code = code_by_residue.get((chain_id, target_residue), '?')
    ca = ca_coordinates(array)
    target_bend = float('nan')
    try:
        target_bend = bend_angle(ca[(chain_id, target_residue - 2)], ca[(chain_id, target_residue)], ca[(chain_id, target_residue + 2)])
    except KeyError:
        pass
    deviations = [np.linalg.norm(ca[key] - original_ca[key]) for key in [(chain_id, residue) for residue in range(window_start, window_end + 1)] if key in ca and key in original_ca]
    metadata_clashes = find_numeric_metric(metadata, {'sidechain_clashes', 'num_sidechain_clashes', 'clashes'})
    chainbreaks = count_ca_chainbreaks(array)
    return {
        'target_sse': target_code,
        'window_sse': ''.join(window_codes),
        'window_helix_fraction': float(window_codes.count('a') / len(window_codes)),
        'target_nonhelical': bool(target_code == 'c'),
        'target_bend_angle_deg': target_bend,
        'window_ca_rmsd_from_original': float(np.sqrt(np.mean(np.square(deviations)))) if deviations else float('nan'),
        'window_ca_max_deviation': float(max(deviations)) if deviations else float('nan'),
        'chainbreaks': int(chainbreaks),
        'metadata_sidechain_clashes': metadata_clashes,
        'quality_pass': bool(chainbreaks == 0 and (metadata_clashes is None or metadata_clashes == 0)),
    }

def json_safe(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, float):
        return None if not math.isfinite(value) else value
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    try:
        json.dumps(value)
        return value
    except TypeError:
        return str(value)

def atom_identity_keys(atom_array):
    array = as_atom_array(atom_array)
    altloc = getattr(array, 'altloc_id', np.full(len(array), ''))
    return [(str(array.chain_id[i]), int(array.res_id[i]), str(array.atom_name[i]), str(altloc[i])) for i in range(len(array))]

def common_topology(atom_arrays):
    arrays = [as_atom_array(array) for array in atom_arrays]
    key_maps = [{key: index for index, key in enumerate(atom_identity_keys(array))} for array in arrays]
    keys = [key for key in atom_identity_keys(arrays[0]) if all(key in mapping for mapping in key_maps[1:])]
    if not keys:
        raise ValueError('The original and generated structures have no common atom topology.')
    indices = [[mapping[key] for key in keys] for mapping in key_maps]
    template = arrays[0][indices[0]].copy()
    coordinates = np.stack([array.coord[index_list] for array, index_list in zip(arrays, indices)], axis=0)
    return template, coordinates, len(keys)

def rigid_superpose(mobile, reference, coordinates):
    mobile_centroid = np.mean(mobile, axis=0)
    reference_centroid = np.mean(reference, axis=0)
    mobile_centered = mobile - mobile_centroid
    reference_centered = reference - reference_centroid
    covariance = mobile_centered.T @ reference_centered
    left, _, right_transpose = np.linalg.svd(covariance)
    rotation = left @ right_transpose
    if np.linalg.det(rotation) < 0.0:
        left[:, -1] *= -1.0
        rotation = left @ right_transpose
    return (coordinates - mobile_centroid) @ rotation + reference_centroid

def choose_alignment_indices(template, coordinates, chain_id, window_start, window_end, mode):
    if mode == 'No alignment':
        return np.array([], dtype=int), 'No alignment'
    if mode not in {'Fixed scaffold outside window', 'All common protein C-alpha'}:
        raise ValueError(f'Unknown ALIGNMENT_MODE: {mode!r}')
    protein_mask = np.asarray(struc.filter_amino_acids(template), dtype=bool)
    atom_names = np.asarray(template.atom_name).astype(str)
    ca_mask = protein_mask & (atom_names == 'CA')
    if mode == 'Fixed scaffold outside window':
        chain_ids = np.asarray(template.chain_id).astype(str)
        residue_ids = np.asarray(template.res_id, dtype=int)
        outside_window = ~((chain_ids == str(chain_id)) & (residue_ids >= int(window_start)) & (residue_ids <= int(window_end)))
        ca_mask &= outside_window
    candidates = np.where(ca_mask)[0]
    if len(candidates):
        finite = np.all(np.isfinite(coordinates[:, candidates, :]), axis=(0, 2))
        candidates = candidates[finite]
    selected_mode = mode
    if len(candidates) < 3 and mode == 'Fixed scaffold outside window':
        fallback = np.where(protein_mask & (atom_names == 'CA'))[0]
        if len(fallback):
            finite = np.all(np.isfinite(coordinates[:, fallback, :]), axis=(0, 2))
            fallback = fallback[finite]
        if len(fallback) >= 3:
            candidates = fallback
            selected_mode = 'All common protein C-alpha (fallback; too few scaffold anchors)'
    if len(candidates) < 3:
        raise ValueError('At least three finite common protein C-alpha atoms are required for alignment.')
    return candidates, selected_mode

def align_multiframe_coordinates(coordinates, template, chain_id, window_start, window_end, mode):
    aligned = np.asarray(coordinates, dtype=float).copy()
    if mode == 'No alignment':
        return aligned, {'mode': 'No alignment', 'atom_count': 0, 'rmsd_angstrom': []}
    indices, selected_mode = choose_alignment_indices(template, aligned, chain_id, window_start, window_end, mode)
    reference = aligned[0, indices].copy()
    rmsds = [0.0]
    for frame_index in range(1, len(aligned)):
        aligned[frame_index] = rigid_superpose(aligned[frame_index, indices], reference, aligned[frame_index])
        anchor_deviation = aligned[frame_index, indices] - reference
        rmsds.append(float(np.sqrt(np.mean(np.sum(anchor_deviation * anchor_deviation, axis=1)))))
    return aligned, {'mode': selected_mode, 'atom_count': int(len(indices)), 'rmsd_angstrom': rmsds}

def write_multiframe_pdb(atom_arrays, path, align_mode, target_chain, window_start, window_end):
    template, coordinates, common_atoms = common_topology(atom_arrays)
    coordinates, alignment_info = align_multiframe_coordinates(coordinates, template, target_chain, window_start, window_end, align_mode)
    pdb = PDBFile()
    pdb.set_structure(struc.from_template(pdb_compatible_array(template), coordinates))
    pdb.write(path)
    return {'common_atoms': int(common_atoms), 'alignment': alignment_info}

def run_rfd3(design_spec, samples, batch_size, seed, step_scale, noise_scale):
    disable_cuequivariance()
    from rfd3.engine import RFD3InferenceConfig, RFD3InferenceEngine
    batches = math.ceil(samples / batch_size)
    config = RFD3InferenceConfig(
        ckpt_path='rfd3',
        diffusion_batch_size=batch_size,
        skip_existing=False,
        prevalidate_inputs=True,
        dump_prediction_metadata_json=True,
        seed=seed,
        specification=design_spec,
        inference_sampler={'step_scale': step_scale, 'noise_scale': noise_scale},
    )
    engine = RFD3InferenceEngine(**config.__dict__)
    outputs_by_example = engine.run(inputs=None, n_batches=batches, out_dir=None)
    outputs = []
    for example_id in sorted(outputs_by_example):
        outputs.extend(outputs_by_example[example_id])
    return outputs[:samples]


In [ ]:
#@title Step 4: Prepare input and build the RFD3 specification
input_source_path = None
if INPUT_MODE == 'Sequence':
    wild_type_sequence = validate_sequence(PROTEIN_SEQUENCE)
    sequence, applied_mutations = apply_mutations(wild_type_sequence, MUTATIONS)
    print(f'Folding {len(sequence)} residues with RF3...')
    original_atom_array = fold_sequence(sequence, example_id='sequence_input')
    input_description = 'RF3-folded sequence'
else:
    if MUTATIONS.strip():
        raise ValueError('For structure mode, upload a structure that already contains the desired mutation and leave MUTATIONS empty. Hand-renaming a residue is not a valid side-chain model.')
    structure_path = UPLOADED_STRUCTURE_PATH
    if structure_path is None:
        raise RuntimeError('No structure is ready. Run Step 2 to upload one PDB/mmCIF file, or enter an existing path in STRUCTURE_PATH.')
    input_source_path = Path(structure_path)
    original_atom_array = load_structure_file(structure_path)
    applied_mutations = []
    input_description = f'Structure input: {structure_path.name}'

original_input_pdb = WORKDIR / 'original_input.pdb'
write_single_pdb(original_atom_array, original_input_pdb)

protein = protein_array(original_atom_array)
protein_records = residue_records(protein)
available_chains = sorted({record.chain_id for record in protein_records})
if TARGET_CHAIN not in available_chains:
    raise ValueError(f'Chain {TARGET_CHAIN!r} was not found. Available protein chains: {available_chains}')
residue_index(protein_records, TARGET_CHAIN, TARGET_RESIDUE)

if WINDOW_MODE == 'Target helix plus padding':
    window_start, window_end, window_source = detect_window(original_atom_array, TARGET_CHAIN, TARGET_RESIDUE, HELIX_PADDING, FALLBACK_HALF_WINDOW)
else:
    window_start, window_end, window_source = int(WINDOW_START), int(WINDOW_END), 'manual'
if window_start > window_end:
    raise ValueError('WINDOW_START must be <= WINDOW_END.')
if not window_start <= TARGET_RESIDUE <= window_end:
    raise ValueError('TARGET_RESIDUE must lie inside the selected window.')
chain_residue_ids = {record.res_id for record in protein_records if record.chain_id == TARGET_CHAIN}
missing_window_residues = [residue for residue in range(window_start, window_end + 1) if residue not in chain_residue_ids]
if missing_window_residues:
    raise ValueError(f'The selected window contains missing/non-contiguous residues: {missing_window_residues}')

# Build the contig from protein residues; RFD3 handles non-protein components as fixed
# partial-diffusion context unless they are explicitly selected otherwise.
all_records = residue_records(original_atom_array, protein_only=True)
fixed_selection = make_fixed_selection(all_records, TARGET_CHAIN, window_start, window_end)
if EXPOSURE_CONDITION == 'Window':
    exposed_selection = f'{TARGET_CHAIN}{window_start}-{window_end}'
elif EXPOSURE_CONDITION == 'Target residue':
    exposed_selection = f'{TARGET_CHAIN}{TARGET_RESIDUE}'
else:
    exposed_selection = None

design_spec = {
    'dialect': 2,
    'input': str(original_input_pdb.resolve()),
    'partial_t': float(PARTIAL_T),
    'select_fixed_atoms': fixed_selection,
    'select_unfixed_sequence': False,
    'extra': {
        'method': 'anchor-preserving local conformational ensemble',
        'target_chain': TARGET_CHAIN,
        'target_residue': int(TARGET_RESIDUE),
        'diffuse_window_start': int(window_start),
        'diffuse_window_end': int(window_end),
        'window_source': window_source,
        'mutations': applied_mutations,
    },
}
if exposed_selection is not None:
    design_spec['select_exposed'] = exposed_selection
if LOOP_BIAS == 'More loops':
    design_spec['is_non_loopy'] = False
elif LOOP_BIAS == 'Fewer loops':
    design_spec['is_non_loopy'] = True

SPEC_PATH = WORKDIR / 'design_spec.json'
SPEC_PATH.write_text(json.dumps(json_safe(design_spec), indent=2) + '\n')
ORIGINAL_CA = ca_coordinates(original_atom_array)
print('Input:', input_description)
print('Applied sequence mutations:', applied_mutations or 'none')
print('Window:', f'{TARGET_CHAIN}{window_start}-{window_end}', f'({window_source})')
print('Fixed scaffold:', fixed_selection)
print('RFD3 specification:', SPEC_PATH)

In [ ]:
#@title Step 5: Generate local variants with RFD3
if N_VARIANTS < 1:
    raise ValueError('N_VARIANTS must be at least 1.')
if DIFFUSION_BATCH_SIZE < 1:
    raise ValueError('DIFFUSION_BATCH_SIZE must be at least 1.')
print(f'Generating {N_VARIANTS} variants with partial_t={PARTIAL_T}...')
rfd3_outputs = run_rfd3(design_spec, N_VARIANTS, DIFFUSION_BATCH_SIZE, RFD3_SEED, STEP_SCALE, NOISE_SCALE)
if not rfd3_outputs:
    raise RuntimeError('RFD3 returned no structures.')
print(f'RFD3 returned {len(rfd3_outputs)} structures.')

VARIANT_ARRAYS = [as_atom_array(output.atom_array) for output in rfd3_outputs]
for index, array in enumerate(VARIANT_ARRAYS, start=1):
    write_single_pdb(array, WORKDIR / f'variant_{index:03d}.pdb')


In [ ]:
#@title Step 6: Analyze variants and build the comparison ensemble
reference_metrics = analyze_structure(original_atom_array, ORIGINAL_CA, {}, TARGET_CHAIN, TARGET_RESIDUE, window_start, window_end)
frame_rows = [{
    'frame': 1, 'label': 'Original', 'sample_id': 'original', 'status': 'Reference',
    **reference_metrics, 'partial_t': 0.0, 'accepted': True,
}]

for index, output in enumerate(rfd3_outputs, start=1):
    metadata = getattr(output, 'metadata', {}) or {}
    metrics = analyze_structure(VARIANT_ARRAYS[index - 1], ORIGINAL_CA, metadata, TARGET_CHAIN, TARGET_RESIDUE, window_start, window_end)
    quality_pass = metrics['quality_pass']
    accepted = bool(quality_pass and (metrics['target_nonhelical'] if REQUIRE_TARGET_NONHELICAL else True))
    metrics.update({
        'frame': index + 1,
        'label': f'Variant {index:02d}',
        'sample_id': str(getattr(output, 'example_id', f'variant_{index:03d}')),
        'status': 'Accepted' if accepted else 'Rejected',
        'partial_t': float(PARTIAL_T),
        'accepted': accepted,
    })
    frame_rows.append(metrics)

FRAME_LABELS = [f"{row['label']} | target SSE={row.get('target_sse', '?')} | window helix={row.get('window_helix_fraction', float('nan')):.2f}" for row in frame_rows]
ALL_MULTIFRAME_PATH = WORKDIR / 'original_plus_variants_multiframe.pdb'
all_frame_info = write_multiframe_pdb([original_atom_array] + VARIANT_ARRAYS, ALL_MULTIFRAME_PATH, ALIGNMENT_MODE, TARGET_CHAIN, window_start, window_end)
common_atom_count = all_frame_info['common_atoms']
alignment_info = all_frame_info['alignment']

accepted_variant_indices = [index for index, row in enumerate(frame_rows[1:]) if row['accepted']]
ACCEPTED_MULTIFRAME_PATH = None
if accepted_variant_indices:
    ACCEPTED_MULTIFRAME_PATH = WORKDIR / 'original_plus_accepted_variants_multiframe.pdb'
    write_multiframe_pdb([original_atom_array] + [VARIANT_ARRAYS[index] for index in accepted_variant_indices], ACCEPTED_MULTIFRAME_PATH, ALIGNMENT_MODE, TARGET_CHAIN, window_start, window_end)

RUN_SUMMARY_PATH = WORKDIR / 'run_summary.json'
run_summary = {
    'input_description': input_description,
    'mutations': applied_mutations,
    'target': f'{TARGET_CHAIN}{TARGET_RESIDUE}',
    'window': f'{TARGET_CHAIN}{window_start}-{window_end}',
    'samples_requested': int(N_VARIANTS),
    'samples_generated': len(rfd3_outputs),
    'samples_accepted': len(accepted_variant_indices),
    'common_atoms_in_multiframe': int(common_atom_count),
    'alignment': alignment_info,
    'design_spec': design_spec,
    'frames': [{**row, 'rfd3_metadata': json_safe(getattr(rfd3_outputs[index - 1], 'metadata', {})) if index > 0 else {}} for index, row in enumerate(frame_rows)],
    'files': {
        'original_input_pdb': str(original_input_pdb),
        'design_spec_json': str(SPEC_PATH),
        'all_frames_pdb': str(ALL_MULTIFRAME_PATH),
        'accepted_frames_pdb': str(ACCEPTED_MULTIFRAME_PATH) if ACCEPTED_MULTIFRAME_PATH else None,
    },
}
RUN_SUMMARY_PATH.write_text(json.dumps(json_safe(run_summary), indent=2) + '\n')

RESULTS_DF = pd.DataFrame(frame_rows)
columns = ['frame', 'label', 'status', 'target_sse', 'window_sse', 'window_helix_fraction', 'target_bend_angle_deg', 'window_ca_rmsd_from_original', 'chainbreaks', 'metadata_sidechain_clashes']
display(RESULTS_DF[[column for column in columns if column in RESULTS_DF]].round(3))
print(f'Accepted variants: {len(accepted_variant_indices)} / {len(rfd3_outputs)}')
print(f'Common atoms retained in the multiframe topology: {common_atom_count}')
print(f"Alignment: {alignment_info['mode']} using {alignment_info['atom_count']} C-alpha anchors")
print('All frames:', ALL_MULTIFRAME_PATH)
if ACCEPTED_MULTIFRAME_PATH:
    print('Original plus accepted frames:', ACCEPTED_MULTIFRAME_PATH)

## How to interpret the table

`window_sse` uses Biotite P-SEA codes (`a` = alpha helix, `c` = coil). A variant passes the geometry check when no adjacent numbered Cα pair is farther than 4.5 Å. If RFD3 reports a clash count, it must be zero; if no clash count is present, clashes are unassessed rather than assumed absent. When enabled, the acceptance rule also requires the target residue to be classified as coil. The viewer still includes every generated variant so you can inspect near-misses and adjust `PARTIAL_T`, loop guidance, or the selected window.

For a mutation series, use the same window, flanking scaffold, sample count, parameter grid, and seed policy for each sequence/structure. Run the wild type and mutant separately, then compare accepted-state frequencies and geometry; those frequencies are protocol-dependent and are not thermodynamic populations.

In [ ]:
#@title Step 7: Inspect aligned models in Mol*
def build_molstar_multiframe_html(pdb_path, frame_labels, height=700):
    pdb_text = Path(pdb_path).read_text()
    payload = {
        'pdb_base64': base64.b64encode(pdb_text.encode('utf-8')).decode('ascii'),
        'labels': list(frame_labels),
    }
    token = uuid.uuid4().hex
    viewer_id = f'local-ensemble-viewer-{token}'
    select_id = f'local-ensemble-select-{token}'
    all_button_id = f'local-ensemble-all-{token}'
    loading_id = f'local-ensemble-loading-{token}'
    html = '''
<style>
.local-ensemble-wrap { font-family: system-ui, -apple-system, Segoe UI, Roboto, Arial; }
.local-ensemble-toolbar { display:flex; align-items:center; gap:10px; margin:8px 0; flex-wrap:wrap; }
.local-ensemble-toolbar select, .local-ensemble-toolbar button { padding:7px 10px; border:1px solid #cbd5e1; border-radius:8px; background:#fff; cursor:pointer; }
.local-ensemble-note { color:#475569; font-size:12px; margin:4px 0 8px; }
#__VIEWER_ID__ { width:100%; height:__HEIGHT__px; position:relative; border:1px solid #dbe3ec; border-radius:12px; overflow:hidden; }
.local-ensemble-loading { position:absolute; inset:0; z-index:5; display:flex; align-items:center; justify-content:center; background:#fff; color:#64748b; letter-spacing:.08em; font-size:12px; }
</style>
<link rel='stylesheet' href='https://cdn.jsdelivr.net/npm/molstar@3/build/viewer/molstar.css'>
<script src='https://cdn.jsdelivr.net/npm/molstar@3/build/viewer/molstar.js'></script>
<div class='local-ensemble-wrap'>
  <div class='local-ensemble-toolbar'>
    <label for='__SELECT_ID__'>Frame:</label>
    <select id='__SELECT_ID__'></select>
    <button id='__ALL_BUTTON_ID__'>Show all models</button>
  </div>
  <div class='local-ensemble-note'>The multiframe PDB is pre-aligned to the original using stable C-alpha anchors outside the selected window. Mol* is loaded with the all-models preset; use the selector for a clean one-frame view or show all models for an overlay comparison.</div>
  <div id='__VIEWER_ID__'><div id='__LOADING_ID__' class='local-ensemble-loading'>LOADING MOL*</div></div>
</div>
<script>
(() => {
  const payload = __PAYLOAD__;
  const viewerId = '__VIEWER_ID__';
  const selectId = '__SELECT_ID__';
  const allButtonId = '__ALL_BUTTON_ID__';
  const loadingId = '__LOADING_ID__';
  const text = new TextDecoder().decode(Uint8Array.from(atob(payload.pdb_base64), c => c.charCodeAt(0)));
  let viewer = null;
  let structures = [];
  const byId = id => document.getElementById(id);
  function fillSelector() {
    const select = byId(selectId);
    if (!select) return;
    payload.labels.forEach((label, index) => {
      const option = document.createElement('option');
      option.value = String(index);
      option.textContent = label;
      select.appendChild(option);
    });
  }
  function setHidden(index) {
    if (!viewer || !structures.length) return;
    structures.forEach((structure, current) => viewer.plugin.state.data.updateCellState(structure.ref, { isHidden: current !== index }));
    if (viewer.plugin.managers.camera.reset) viewer.plugin.managers.camera.reset();
  }
  function showAll() {
    if (!viewer || !structures.length) return;
    structures.forEach(structure => viewer.plugin.state.data.updateCellState(structure.ref, { isHidden: false }));
    if (viewer.plugin.managers.camera.reset) viewer.plugin.managers.camera.reset();
  }
  async function init() {
    try {
      viewer = await molstar.Viewer.create(viewerId, { layoutIsExpanded:false, layoutShowControls:true, layoutShowRemoteState:false, layoutShowSequence:true, layoutShowLog:false });
      const data = await viewer.plugin.builders.data.rawData({ data: text, label: 'Original plus local variants' });
      const trajectory = await viewer.plugin.builders.structure.parseTrajectory(data, 'pdb');
      const result = await viewer.plugin.builders.structure.hierarchy.applyPreset(trajectory, 'all-models', { useDefaultIfSingleModel:true, showUnitcell:false, representationPreset:'auto' });
      structures = result && result.structures ? result.structures : [];
      fillSelector();
      const select = byId(selectId);
      if (select) select.addEventListener('change', () => setHidden(Number(select.value)));
      const allButton = byId(allButtonId);
      if (allButton) allButton.addEventListener('click', showAll);
      showAll();
      const loading = byId(loadingId);
      if (loading) loading.style.display = 'none';
    } catch (error) {
      console.error('Mol* multiframe load failed:', error);
      const loading = byId(loadingId);
      if (loading) loading.textContent = 'Mol* failed to load; the PDB is still available below.';
    }
  }
  function waitForMolstar(tries = 0) {
    if (typeof molstar !== 'undefined' && molstar.Viewer) { init(); return; }
    if (tries < 60) { setTimeout(() => waitForMolstar(tries + 1), 200); return; }
    const loading = byId(loadingId);
    if (loading) loading.textContent = 'Mol* JavaScript did not load; use the PDB link below.';
  }
  waitForMolstar();
})();
</script>
'''
    return (html.replace('__PAYLOAD__', json.dumps(payload))
                .replace('__VIEWER_ID__', viewer_id)
                .replace('__SELECT_ID__', select_id)
                .replace('__ALL_BUTTON_ID__', all_button_id)
                .replace('__LOADING_ID__', loading_id)
                .replace('__HEIGHT__', str(int(height))))

display(HTML(build_molstar_multiframe_html(ALL_MULTIFRAME_PATH, FRAME_LABELS)))

In [ ]:
#@title Step 8: Download final files
import zipfile
from IPython.display import clear_output
from google.colab import files, output as colab_output
import ipywidgets as widgets

colab_output.enable_custom_widget_manager()

RESULTS_ZIP_PATH = WORKDIR / 'local_conformational_ensemble_results.zip'
bundle_entries = [
    ('inputs/original_input.pdb', original_input_pdb),
    ('ensembles/original_plus_variants_multiframe.pdb', ALL_MULTIFRAME_PATH),
    ('metadata/design_spec.json', SPEC_PATH),
    ('metadata/run_summary.json', RUN_SUMMARY_PATH),
]
if input_source_path is not None and Path(input_source_path).is_file():
    bundle_entries.append((f'inputs/uploaded_original/{Path(input_source_path).name}', Path(input_source_path)))
if ACCEPTED_MULTIFRAME_PATH is not None:
    bundle_entries.append(('ensembles/original_plus_accepted_variants_multiframe.pdb', ACCEPTED_MULTIFRAME_PATH))
for variant_index in range(1, len(VARIANT_ARRAYS) + 1):
    variant_path = WORKDIR / f'variant_{variant_index:03d}.pdb'
    bundle_entries.append((f'individual_variants/{variant_path.name}', variant_path))
with zipfile.ZipFile(RESULTS_ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as result_zip:
    for archive_name, file_path in bundle_entries:
        if Path(file_path).is_file():
            result_zip.write(file_path, arcname=archive_name)

download_items = [
    ('Download all results (.zip)', RESULTS_ZIP_PATH),
    ('Download comparison ensemble (.pdb)', ALL_MULTIFRAME_PATH),
    ('Download processed original input (.pdb)', original_input_pdb),
    ('Download run summary (.json)', RUN_SUMMARY_PATH),
    ('Download RFD3 specification (.json)', SPEC_PATH),
]
if ACCEPTED_MULTIFRAME_PATH is not None:
    download_items.insert(2, ('Download accepted ensemble (.pdb)', ACCEPTED_MULTIFRAME_PATH))

download_status = widgets.Output()
download_buttons = []
for item_index, (label, file_path) in enumerate(download_items):
    button = widgets.Button(description=label, button_style='primary' if item_index == 0 else '', layout=widgets.Layout(width='360px'))
    def make_download_handler(path):
        def handle_download(_button):
            with download_status:
                clear_output(wait=True)
                if not Path(path).is_file():
                    print(f'File not found: {path}')
                    return
                print(f'Starting download: {Path(path).name}')
            try:
                files.download(str(path))
            except Exception as exc:
                with download_status:
                    clear_output(wait=True)
                    print(f'Download failed: {exc}')
        return handle_download
    button.on_click(make_download_handler(file_path))
    download_buttons.append(button)

display(widgets.HTML('<h3>Download your results</h3><p>Click a button to save that file to your computer. The ZIP contains the source input (when applicable), original structure, every generated variant, comparison ensembles, and JSON records.</p>'))
display(widgets.VBox(download_buttons))
display(download_status)